# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Mã nguồn dưới đây đã được hoàn thiện chi tiết để chạy trên Google Colab T4 GPU.

## 0. Môi trường (Colab T4)

In [ ]:
# 1. Mount Google Drive để sao lưu trọng số an toàn (tránh mất checkpoint khi session bị ngắt)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/lab_day10_backup

# 2. Kiểm tra GPU và cài đặt thư viện
!nvidia-smi -L
!pip -q install -r requirements-colab.txt
!python -m pytest -q

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
import json
from pathlib import Path
from PIL import Image

def read_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

meta = json.loads(Path('data/splits/meta.json').read_text(encoding='utf-8'))
print(f"Dataset version: {meta.get('dataset_version')}")

for split in ['train', 'val', 'test']:
    entries = read_jsonl(f'data/splits/{split}.jsonl')
    boxes_count = sum(len(e['objects']) for e in entries)
    print(f"Split {split}: {len(entries)} ảnh, {boxes_count} boxes")

# Kiểm tra rò rỉ session
train_sessions = {e['session'] for e in read_jsonl('data/splits/train.jsonl')}
val_sessions = {e['session'] for e in read_jsonl('data/splits/val.jsonl')}
test_sessions = {e['session'] for e in read_jsonl('data/splits/test.jsonl')}
assert len(train_sessions & val_sessions) == 0, 'Trùng session giữa train và val!'
assert len(train_sessions & test_sessions) == 0, 'Trùng session giữa train và test!'
assert len(val_sessions & test_sessions) == 0, 'Trùng session giữa val và test!'
print('Xác nhận: Hoàn toàn không có session nào bị rò rỉ giữa các split!')

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import json

SYSTEM_PROMPT = (
    "Detect all road damage instances (pothole, crack, manhole) in this image. "
    "Return ONLY a JSON list of objects. Each object must have 'label' (one of: pothole, crack, manhole) "
    "and 'bbox_2d': [x1, y1, x2, y2] on a 0-1000 integer normalized scale. "
    "Do not output markdown code blocks or text outside the JSON list."
)

def yolo_to_box(label, yolo):
    xc, yc, w, h = yolo
    x1 = int(round(max(0.0, min(1.0, xc - w / 2)) * 1000))
    y1 = int(round(max(0.0, min(1.0, yc - h / 2)) * 1000))
    x2 = int(round(max(0.0, min(1.0, xc + w / 2)) * 1000))
    y2 = int(round(max(0.0, min(1.0, yc + h / 2)) * 1000))
    x2 = max(x1 + 1, x2)
    y2 = max(y1 + 1, y2)
    return {'label': label, 'bbox_2d': [x1, y1, x2, y2]}

def objects_to_answer(objects):
    boxes = [yolo_to_box(o['label'], o['yolo']) for o in objects]
    # Sắp xếp trên -> dưới (y1), trái -> phải (x1) để output ổn định
    boxes.sort(key=lambda b: (b['bbox_2d'][1], b['bbox_2d'][0]))
    return json.dumps(boxes, separators=(',', ':'))

# Test mẫu
sample_test = [{'label': 'pothole', 'yolo': [0.5, 0.5, 0.2, 0.1]}]
print('Test objects_to_answer:', objects_to_answer(sample_test))

## 3. E0: zero-shot

In [ ]:
import os
import torch
from transformers import BitsAndBytesConfig, AutoProcessor, Qwen3VLForConditionalGeneration
from PIL import Image
from tqdm import tqdm

os.makedirs('work', exist_ok=True)
os.makedirs('report/evidence', exist_ok=True)

model_id = 'Qwen/Qwen3-VL-4B-Instruct'

# Giữ visual và lm_head ở float16 để không làm hỏng weights projector ở E2/E3
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=['visual', 'lm_head']
)

processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = 'left'

model_e0 = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map={'': 0}
)
model_e0.eval()

test_entries = read_jsonl('data/splits/test.jsonl')
e0_results = []
batch_size = 4

with torch.inference_mode():
    for i in tqdm(range(0, len(test_entries), batch_size), desc='Suy luận E0 Zero-shot'):
        batch = test_entries[i:i + batch_size]
        images = [Image.open(f"data/{e['image_path']}").convert('RGB') for e in batch]
        messages_batch = [
            [{'role': 'user', 'content': [
                {'type': 'image', 'image': img},
                {'type': 'text', 'text': SYSTEM_PROMPT}
            ]}] for img in images
        ]
        prompts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages_batch]
        inputs = processor(text=prompts, images=images, return_tensors='pt', padding=True).to('cuda')
        out = model_e0.generate(**inputs, max_new_tokens=256, do_sample=False)
        prompt_len = inputs.input_ids.shape[1]
        decoded = processor.batch_decode(out[:, prompt_len:], skip_special_tokens=True)
        for e, raw in zip(batch, decoded):
            e0_results.append({'image_id': e['image_id'], 'raw': raw})

with open('work/e0.jsonl', 'w', encoding='utf-8') as f:
    for row in e0_results:
        f.write(json.dumps(row) + '\n')

del model_e0
torch.cuda.empty_cache()
print('Hoàn tất suy luận E0!')

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from torch.utils.data import Dataset, DataLoader

class RoadDataset(Dataset):
    def __init__(self, entries, img_prefix='data/'):
        self.entries = entries
        self.img_prefix = img_prefix

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        e = self.entries[idx]
        img = Image.open(f"{self.img_prefix}{e['image_path']}").convert('RGB')
        answer = objects_to_answer(e['objects'])
        return img, SYSTEM_PROMPT, answer

def collate_train_batch(batch, processor):
    images = [item[0] for item in batch]
    prompts = [item[1] for item in batch]
    answers = [item[2] for item in batch]

    user_msgs = [[{'role': 'user', 'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': p}]}]
                 for img, p in zip(images, prompts)]
    full_msgs = [[
        {'role': 'user', 'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': p}]},
        {'role': 'assistant', 'content': [{'type': 'text', 'text': ans}]}
    ] for img, p, ans in zip(images, prompts, answers)]

    prompt_texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in user_msgs]
    full_texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in full_msgs]

    inputs = processor(text=full_texts, images=images, return_tensors='pt', padding=True)
    prompt_inputs = processor(text=prompt_texts, images=images, return_tensors='pt', padding=True)

    labels = inputs.input_ids.clone()
    prompt_len = prompt_inputs.input_ids.shape[1]
    # Mask toàn bộ prompt và image token bằng -100 (chỉ học phần sinh JSON)
    labels[:, :prompt_len] = -100
    labels[inputs.attention_mask == 0] = -100
    inputs['labels'] = labels
    return inputs

train_entries = read_jsonl('data/splits/train.jsonl')
train_dataset = RoadDataset(train_entries)
sample_loader = DataLoader(train_dataset, batch_size=2, shuffle=False, collate_fn=lambda b: collate_train_batch(b, processor))
sample_batch = next(iter(sample_loader))
print('Input IDs shape:', sample_batch.input_ids.shape)
print('Labels shape:', sample_batch.labels.shape)
print(f"Số token tính loss của mẫu 0: {(sample_batch.labels[0] != -100).sum().item()}")

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
from peft import LoraConfig, get_peft_model

def apply_setup(model, setup_name, rank=16, alpha=32):
    for param in model.parameters():
        param.requires_grad = False
    model.enable_input_require_grads()

    target_modules = [
        r'model\.language_model\.layers\.\d+\.self_attn\.(q|k|v|o)_proj',
        r'model\.language_model\.layers\.\d+\.mlp\.(gate|up|down)_proj'
    ]
    if setup_name == 'E3':
        target_modules.extend([
            r'model\.visual\.blocks\.\d+\.attn\.(qkv|proj)',
            r'model\.visual\.blocks\.\d+\.mlp\.(linear_fc1|linear_fc2)'
        ])

    peft_config = LoraConfig(
        r=rank,
        lora_alpha=alpha,
        lora_dropout=0.05,
        target_modules=target_modules,
        bias='none'
    )
    model = get_peft_model(model, peft_config)

    # Nếu là E2 hoặc E3: mở đầy đủ projector (merger) và đưa về float32 để tránh tràn số
    if setup_name in ['E2', 'E3']:
        for name, param in model.named_parameters():
            if 'visual.merger' in name or 'visual.deepstack_merger_list' in name:
                param.data = param.data.to(torch.float32)
                param.requires_grad = True

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'[{setup_name}] Trainable params: {trainable_params:,}')
    return model, trainable_params

## 6. Vòng huấn luyện

In [ ]:
import time
from torch.cuda.amp import GradScaler, autocast
from transformers import get_cosine_schedule_with_warmup

def run_training(setup_name, rank=16, alpha=32, max_steps=300, lr=2e-4, batch_size=2, grad_accum=4, fraction=1.0):
    print(f'\n=================== BẮT ĐẦU HUẤN LUYỆN {setup_name} ===================')
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    model = Qwen3VLForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map={'': 0}
    )
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    model.config.use_cache = False

    model, trainable_params = apply_setup(model, setup_name, rank=rank, alpha=alpha)

    entries = train_entries
    if fraction < 1.0:
        entries = entries[:int(len(entries) * fraction)]
        print(f'Sử dụng {fraction*100:.0f}% dữ liệu: {len(entries)} ảnh')

    ds = RoadDataset(entries)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, collate_fn=lambda b: collate_train_batch(b, processor))

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=lr, weight_decay=0.0)
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(max_steps * 0.05), num_training_steps=max_steps)
    scaler = GradScaler('cuda')

    step = 0
    model.train()
    pbar = tqdm(total=max_steps, desc=f'Training {setup_name}')

    while step < max_steps:
        for b_idx, batch in enumerate(loader):
            batch = {k: v.to('cuda') for k, v in batch.items()}
            with autocast(dtype=torch.float16):
                loss = model(**batch).loss / grad_accum

            scaler.scale(loss).backward()

            if (b_idx + 1) % grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()
                step += 1
                pbar.update(1)
                pbar.set_postfix({'loss': f"{loss.item() * grad_accum:.4f}"})
                if step >= max_steps:
                    break
    pbar.close()

    train_minutes = round((time.time() - start_time) / 60.0, 2)
    peak_vram_gb = round(torch.cuda.max_memory_allocated() / (1024 ** 3), 2)
    print(f'Hoàn thành {setup_name}: {train_minutes} phút, Peak VRAM: {peak_vram_gb} GB')

    save_dir = f'work/{setup_name.lower()}_weights'
    os.makedirs(save_dir, exist_ok=True)
    model.save_pretrained(save_dir)

    # Lưu riêng weights của Projector nếu có (E2, E3)
    if setup_name in ['E2', 'E3']:
        proj_weights = {
            name: param.data.cpu() for name, param in model.named_parameters()
            if ('visual.merger' in name or 'visual.deepstack_merger_list' in name) and param.requires_grad
        }
        torch.save(proj_weights, f'{save_dir}/projector.pt')
        print(f'Đã lưu projector weights tại {save_dir}/projector.pt')

    # Sao lưu sang Google Drive an toàn
    !cp -r {save_dir} /content/drive/MyDrive/lab_day10_backup/ 2>/dev/null || true

    del model
    del optimizer
    del scheduler
    torch.cuda.empty_cache()

    return {
        'trainable_params': trainable_params,
        'peak_vram_gb': peak_vram_gb,
        'train_minutes': train_minutes
    }

## 7. Suy luận và chấm E1–E3

In [ ]:
from peft import PeftModel

def evaluate_and_score(setup_name, meta_info, coord='norm1000'):
    save_dir = f'work/{setup_name.lower()}_weights'
    print(f'Đang nạp checkpoint {setup_name} từ {save_dir}...')

    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map={'': 0}
    )
    eval_model = PeftModel.from_pretrained(base_model, save_dir)

    proj_path = f'{save_dir}/projector.pt'
    if os.path.exists(proj_path):
        proj_state = torch.load(proj_path)
        eval_model.load_state_dict(proj_state, strict=False)
        print('Đã nạp thành công weights projector!')

    eval_model.eval()
    preds = []

    with torch.inference_mode():
        for i in tqdm(range(0, len(test_entries), 4), desc=f'Inference {setup_name}'):
            batch = test_entries[i:i + 4]
            images = [Image.open(f"data/{e['image_path']}").convert('RGB') for e in batch]
            messages_batch = [
                [{'role': 'user', 'content': [
                    {'type': 'image', 'image': img},
                    {'type': 'text', 'text': SYSTEM_PROMPT}
                ]}] for img in images
            ]
            prompts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages_batch]
            inputs = processor(text=prompts, images=images, return_tensors='pt', padding=True).to('cuda')
            out = eval_model.generate(**inputs, max_new_tokens=256, do_sample=False)
            prompt_len = inputs.input_ids.shape[1]
            decoded = processor.batch_decode(out[:, prompt_len:], skip_special_tokens=True)
            for e, raw in zip(batch, decoded):
                preds.append({'image_id': e['image_id'], 'raw': raw})

    pred_file = f'work/{setup_name.lower()}.jsonl'
    with open(pred_file, 'w', encoding='utf-8') as f:
        for r in preds:
            f.write(json.dumps(r) + '\n')

    del eval_model
    del base_model
    torch.cuda.empty_cache()

    out_evidence = f'report/evidence/{setup_name.lower()}.json'
    cmd = (
        f"python scripts/score.py --pred {pred_file} --name {setup_name} --out {out_evidence} --coord {coord} "
        f"--meta trainable_params={meta_info['trainable_params']} "
        f"--meta peak_vram_gb={meta_info['peak_vram_gb']} "
        f"--meta train_minutes={meta_info['train_minutes']}"
    )
    !{cmd}

# 1. Chạy E1 (LoRA LLM)
meta_e1 = run_training('E1')
evaluate_and_score('E1', meta_e1)

# 2. Chạy E2 (LoRA LLM + Projector)
meta_e2 = run_training('E2')
evaluate_and_score('E2', meta_e2)

# 3. Chạy E3 (LoRA LLM + Projector + LoRA ViT)
meta_e3 = run_training('E3')
evaluate_and_score('E3', meta_e3)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4: Ablation study thử nghiệm LoRA rank r=4, alpha=8 (đổi 1 biến so với setup gốc)
meta_e4_rank4 = run_training('E4_rank_4', rank=4, alpha=8)
evaluate_and_score('E4_rank_4', meta_e4_rank4)

# Kiểm tra toàn diện theo quy chuẩn bài nộp
!python scripts/check_submission.py
print('Hoàn thành toàn bộ thí nghiệm Lab Day 10!')